<a href="https://colab.research.google.com/github/jyryu3161/2026CADD/blob/main/week2_Kinase_Key_Residue_Finder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Standardized kinase key-residue finder
## PDB → UniProt kinase-domain annotation → motif QC → kinase-reference mapping → 3D view

Example: **8VF6 (human STK33)**

### Why this version is better
The previous notebook used a manually entered PKA range (`40–300`).  
This notebook removes that target-independent hard coding.

Instead:

1. Obtain the target UniProt accession from the PDB/mmCIF metadata.
2. Fetch the **target protein sequence and annotated protein-kinase domain** from UniProt.
3. Map the experimental PDB chain onto that UniProt kinase domain.
4. Detect conserved kinase motifs directly:
   - gly-rich loop
   - β3 Lys / VAIK-like motif
   - HRD
   - DFG
   - APE
5. Fetch the kinase-domain annotation of a canonical reference kinase (PRKACA) automatically.
6. Use kinase-domain alignment only for landmarks that are difficult to define from a single sequence motif:
   - αC Glu
   - gatekeeper
   - hinge positions
7. Cross-check topology and display the residues in 3D.

### Important interpretation
- **Ligand is not required** to identify the kinase hinge region, gatekeeper, HRD, DFG, catalytic Lys, etc.
- Without a ligand, we identify the **candidate hinge backbone region**.
- The exact hinge residue(s) used as hydrogen-bond partners by a particular inhibitor require an inhibitor pose.

This is a practical hybrid workflow close to how kinase annotation is normally done:
**domain annotation + kinase motifs + profile/reference alignment + structural QC**.

In [ ]:
!pip -q install biopython pandas requests py3Dmol

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 28.6 MB/s eta 0:00:00


In [ ]:
# ==============================
# User input
# ==============================
PDB_ID = "8VF6"
CHAIN_ID = "A"

# Canonical reference kinase used only to standardize difficult landmarks.
# PRKACA is a classical Hanks-type Ser/Thr kinase.
REFERENCE_UNIPROT = "P17612"

# Protein-only visualization; ligand atoms are ignored.
IGNORE_LIGANDS = True

In [ ]:
import re, json, requests, pandas as pd
from Bio.PDB import MMCIFParser, PDBIO, Select
from Bio import Align
from Bio.Align import substitution_matrices
import py3Dmol

AA3_TO_1 = {
    "ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E","GLY":"G",
    "HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P","SER":"S",
    "THR":"T","TRP":"W","TYR":"Y","VAL":"V","SEC":"U","PYL":"O"
}

def get_json(url):
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json()

def get_text(url):
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.text

## 1. Download PDB/mmCIF and obtain the target UniProt accession automatically

In [ ]:
cif_url = f"https://files.rcsb.org/download/{PDB_ID}.cif"
cif_text = get_text(cif_url)
open(f"{PDB_ID}.cif","w").write(cif_text)

# RCSB entry metadata
entry = get_json(f"https://data.rcsb.org/rest/v1/core/entry/{PDB_ID}")
entity_ids = entry["rcsb_entry_container_identifiers"]["polymer_entity_ids"]

# Find polymer entity containing requested auth chain and UniProt accession
target_uniprot = None
target_entity = None
for eid in entity_ids:
    pe = get_json(f"https://data.rcsb.org/rest/v1/core/polymer_entity/{PDB_ID}/{eid}")
    ids = pe.get("rcsb_polymer_entity_container_identifiers", {})
    auth_chains = ids.get("auth_asym_ids", [])
    if CHAIN_ID in auth_chains:
        target_entity = pe
        refs = ids.get("reference_sequence_identifiers", []) or []
        for ref in refs:
            if ref.get("database_name") == "UniProt":
                target_uniprot = ref.get("database_accession")
                break
        if target_uniprot:
            break

print("PDB:", PDB_ID)
print("Chain:", CHAIN_ID)
print("Target UniProt:", target_uniprot)
assert target_uniprot is not None, "Could not resolve UniProt accession from RCSB."

PDB: 8VF6
Chain: A
Target UniProt: Q9BYT3


## 2. Read the experimental chain sequence and PDB residue numbering

In [ ]:
parser = MMCIFParser(QUIET=True)
structure = parser.get_structure("target", f"{PDB_ID}.cif")
chain = structure[0][CHAIN_ID]

pdb_seq = []
pdb_residues = []
for res in chain:
    if res.id[0] != " ":
        continue
    aa = AA3_TO_1.get(res.resname.upper())
    if aa:
        pdb_seq.append(aa)
        pdb_residues.append({
            "idx": len(pdb_seq)-1,
            "aa": aa,
            "resnum": res.id[1],
            "icode": (res.id[2] or "").strip(),
            "resname3": res.resname.upper()
        })

pdb_seq = "".join(pdb_seq)
print("Observed protein residues in structure:", len(pdb_seq))
print(pdb_seq[:100] + " ...")

Observed protein residues in structure: 270
KVPHIRIENGAAIEEIYTFGRILGKGSFGIVIEATDKETETKWAIKKVNKEKAGSSAVKLLEREVNILKSVKHEHIIHLEQVFETPKKMYLVMELCEDGE ...


## 3. Fetch UniProt sequence and kinase-domain boundary automatically

No `40–300` or other manually entered domain coordinates are used here.

The notebook searches UniProt features whose annotation indicates a **protein kinase** domain.

In [ ]:
def fetch_uniprot(accession):
    return get_json(f"https://rest.uniprot.org/uniprotkb/{accession}.json")

def extract_sequence(up):
    return up["sequence"]["value"]

def pos_value(x):
    # UniProt JSON can encode position as {"value": 123}
    if x is None:
        return None
    if isinstance(x, int):
        return x
    if isinstance(x, dict):
        return x.get("value")
    return None

def find_kinase_domain(up):
    candidates = []
    for f in up.get("features", []):
        desc = (f.get("description") or "").lower()
        ftype = (f.get("type") or "").lower()
        if ftype == "domain" and ("protein kinase" in desc or desc == "kinase"):
            loc = f.get("location", {})
            start = pos_value(loc.get("start"))
            end = pos_value(loc.get("end"))
            if start and end:
                candidates.append((start, end, f.get("description","")))
    # fallback: any domain with kinase in description
    if not candidates:
        for f in up.get("features", []):
            desc = (f.get("description") or "").lower()
            if "kinase" in desc:
                loc = f.get("location", {})
                start = pos_value(loc.get("start"))
                end = pos_value(loc.get("end"))
                if start and end:
                    candidates.append((start, end, f.get("description","")))
    return candidates

target_up = fetch_uniprot(target_uniprot)
target_full_seq = extract_sequence(target_up)
target_domains = find_kinase_domain(target_up)

print("Target protein length:", len(target_full_seq))
print("Kinase-domain candidates from UniProt:")
for x in target_domains:
    print(x)

assert target_domains, "No kinase-domain annotation found in UniProt."
TARGET_DOMAIN_START, TARGET_DOMAIN_END, TARGET_DOMAIN_DESC = target_domains[0]
target_domain_seq = target_full_seq[TARGET_DOMAIN_START-1:TARGET_DOMAIN_END]

print("\nSelected domain:", TARGET_DOMAIN_START, "-", TARGET_DOMAIN_END, TARGET_DOMAIN_DESC)
print("Domain length:", len(target_domain_seq))

Target protein length: 514
Kinase-domain candidates from UniProt:
(116, 381, 'Protein kinase')

Selected domain: 116 - 381 Protein kinase
Domain length: 266


## 4. Map experimental PDB residues onto the target UniProt sequence

This step is important because crystallographic constructs can be truncated and PDB numbering does not always equal UniProt numbering.

In [ ]:
def make_aligner():
    a = Align.PairwiseAligner()
    a.mode = "global"
    a.substitution_matrix = substitution_matrices.load("BLOSUM62")
    a.open_gap_score = -10
    a.extend_gap_score = -0.5
    return a

aligner = make_aligner()
pdb_to_up_al = aligner.align(target_full_seq, pdb_seq)[0]

# Map UniProt 0-based index -> PDB-chain 0-based index
up_to_pdb = {}
up_blocks, pdb_blocks = pdb_to_up_al.aligned
for (us,ue),(ps,pe) in zip(up_blocks,pdb_blocks):
    n = min(ue-us, pe-ps)
    for k in range(n):
        up_to_pdb[us+k] = ps+k

print(pdb_to_up_al)

target            0 MADSGLDKKSTKCPDCSSASQKDVLCVCSSKTRVPPVLVVEMSQTSSIGSAESLISLERK
                  0 ------------------------------------------------------------
query             0 ------------------------------------------------------------

target           60 KEKNINRDITSRKDLPSRTSNVERKASQQQWGRGNFTEGKVPHIRIENGAAIEEIYTFGR
                 60 ---------------------------------------|||||||||||||||||||||
query             0 ---------------------------------------KVPHIRIENGAAIEEIYTFGR

target          120 ILGKGSFGIVIEATDKETETKWAIKKVNKEKAGSSAVKLLEREVNILKSVKHEHIIHLEQ
                120 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query            21 ILGKGSFGIVIEATDKETETKWAIKKVNKEKAGSSAVKLLEREVNILKSVKHEHIIHLEQ

target          180 VFETPKKMYLVMELCEDGELKEILDRKGHFSENETRWIIQSLASAIAYLHNNDIVHRDLK
                180 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query            81 VFETPKKMYLVMELCEDGELKEILDRKGHFSENETRWIIQSLASAIAYLHNNDIVHRDLK

target          240 LENI

In [ ]:
def pdb_label(pdb_idx):
    r = pdb_residues[pdb_idx]
    return f"{r['aa']}{r['resnum']}{r['icode']}"

def up_label(up_idx0):
    return f"{target_full_seq[up_idx0]}{up_idx0+1}"

def domain_index_to_up(domain_idx0):
    return (TARGET_DOMAIN_START - 1) + domain_idx0

def domain_index_to_pdb(domain_idx0):
    return up_to_pdb.get(domain_index_to_up(domain_idx0))

## 5. Direct kinase-motif detection in the target kinase domain

These motifs are more reliable than transferring every position from another kinase.

In [ ]:
def find_all(seq, regex):
    return [(m.start(), m.group()) for m in re.finditer(regex, seq)]

motif_hits = {
    "HRD": find_all(target_domain_seq, r"HRD"),
    "DFG": find_all(target_domain_seq, r"DFG"),
    "APE": find_all(target_domain_seq, r"APE"),
    # Gly-rich loop: flexible GXGXXG-like pattern
    "Gly-rich loop": find_all(target_domain_seq[:90], r"G[A-Z]G[A-Z]{1,3}G"),
    # Hanks subdomain II motif commonly VAIK / VAVK / related
    "VAIK-like": find_all(target_domain_seq[:140], r"[VILM][A-Z]{1,2}[VILM]K"),
}

for name,hits in motif_hits.items():
    print("\\n", name)
    if not hits:
        print("  no direct hit")
    for i,s in hits:
        ui = domain_index_to_up(i)
        pi = domain_index_to_pdb(i)
        print(" ", i, s, "| UniProt", up_label(ui), "| PDB", pdb_label(pi) if pi is not None else "not observed")

\n HRD
  120 HRD | UniProt H236 | PDB H236
\n DFG
  149 DFG | UniProt D265 | PDB D265
\n APE
  175 APE | UniProt A291 | PDB A291
\n Gly-rich loop
  7 GKGSFG | UniProt G123 | PDB G123
\n VAIK-like
  48 VNILK | UniProt V164 | PDB V164
  128 IMVK | UniProt I244 | PDB I244


## 6. Automatically obtain the reference kinase domain from UniProt

The reference kinase is used only to assign conserved **topological landmarks**
that are not uniquely encoded by a short sequence motif, particularly the
gatekeeper and hinge.

Again, the reference domain boundary is taken from UniProt; it is not typed manually.

In [ ]:
ref_up = fetch_uniprot(REFERENCE_UNIPROT)
ref_full_seq = extract_sequence(ref_up)
ref_domains = find_kinase_domain(ref_up)
assert ref_domains, "No reference kinase-domain annotation found."

REF_DOMAIN_START, REF_DOMAIN_END, REF_DOMAIN_DESC = ref_domains[0]
ref_domain_seq = ref_full_seq[REF_DOMAIN_START-1:REF_DOMAIN_END]

print("Reference:", REFERENCE_UNIPROT)
print("Reference kinase domain:", REF_DOMAIN_START, "-", REF_DOMAIN_END, REF_DOMAIN_DESC)
print("Reference domain length:", len(ref_domain_seq))

Reference: P17612
Reference kinase domain: 44 - 298 Protein kinase
Reference domain length: 255


### Reference landmarks

For PRKACA, classical structural landmarks have conventional residue numbers:

- catalytic Lys: K72
- αC Glu: E91
- gatekeeper: M120
- hinge: residues immediately following the gatekeeper, around 121–123
- HRD: 164–166
- DFG: 184–186
- APE: 206–208

Only these **landmark residue numbers** are curated.  
The reference *domain boundary* itself is fetched automatically from UniProt.

In [ ]:
REFERENCE_LANDMARKS = {
    "Catalytic Lys": [72],
    "alphaC Glu": [91],
    "Gatekeeper": [120],
    "Hinge": [121,122,123],
    "HRD": [164,165,166],
    "DFG": [184,185,186],
    "APE": [206,207,208],
}

# kinase-domain-to-kinase-domain alignment
dom_al = aligner.align(ref_domain_seq, target_domain_seq)[0]

refdom_to_tgtdom = {}
rb,tb = dom_al.aligned
for (rs,re),(ts,te) in zip(rb,tb):
    n=min(re-rs,te-ts)
    for k in range(n):
        refdom_to_tgtdom[rs+k]=ts+k

print(dom_al)

target            0 FERIKTLGTGSFGRVMLVKHKETGNHYAMKILDKQK----VVKLKQIEHTLNEKRILQAV
                  0 ......||.||||.|.....|||....|.|...|.|----.|||--.|...|---||..|
query             0 YTFGRILGKGSFGIVIEATDKETETKWAIKKVNKEKAGSSAVKL--LEREVN---ILKSV

target           56 NFPFLVKLEFSFKDNSNLYMVMEYVPGGEMFSHLRRIGRFSEPHARFYAAQIVLTFEYLH
                 60 .......||..|......|.|||....||....|.|.|.|||...|...........|||
query            55 KHEHIIHLEQVFETPKKMYLVMELCEDGELKEILDRKGHFSENETRWIIQSLASAIAYLH

target          116 SLDLIYRDLKPENL-----LIDQQG----YIQVTDFGFAKRVKGRTWTL----CGTPEYL
                120 ..|...||||.||.-----|||...----.|.|||||.|.....|....----||||.|.
query           115 NNDIVHRDLKLENIMVKSSLIDDNNEINLNIKVTDFGLAVKKQSRSEAMLQATCGTPIYM

target          163 APEIILSKGYNKAVDWWALGVLIYEMAAGYPPFFADQPIQIYEKIVSGKVRFP----SHF
                180 |||.|....|....|.|..||..|....|.|||.|.......|.|..|...|.----...
query           175 APEVISAHDYSQQCDIWSIGVVMYMLLRGEPPFLASSEEKLFELIRKGELHFENAVWNSI

target          219 SSDL

In [ ]:
def ref_resnum_to_refdom_idx(resnum):
    return resnum - REF_DOMAIN_START

features = {}
rows=[]

for feat, nums in REFERENCE_LANDMARKS.items():
    tgt_idxs=[]
    for rr in nums:
        ridx=ref_resnum_to_refdom_idx(rr)
        tidx=refdom_to_tgtdom.get(ridx)
        if tidx is not None:
            tgt_idxs.append(tidx)
    features[feat]=tgt_idxs

    up_labels=[]
    pdb_labels=[]
    for i in tgt_idxs:
        ui=domain_index_to_up(i)
        pi=domain_index_to_pdb(i)
        up_labels.append(up_label(ui))
        pdb_labels.append(pdb_label(pi) if pi is not None else "not observed")
    rows.append({
        "Feature":feat,
        "Target UniProt":", ".join(up_labels),
        "PDB residue":", ".join(pdb_labels),
        "Assignment":"kinase-domain alignment"
    })

pd.DataFrame(rows)

,Feature,Target UniProt,PDB residue,Assignment
0,Catalytic Lys,I144,I144,kinase-domain alignment
1,alphaC Glu,N165,N165,kinase-domain alignment
2,Gatekeeper,V191,V191,kinase-domain alignment
3,Hinge,"M192, E193, L194","M192, E193, L194",kinase-domain alignment
4,HRD,"V235, H236, R237","V235, H236, R237",kinase-domain alignment
5,DFG,"T264, D265, F266","T264, D265, F266",kinase-domain alignment
6,APE,"M290, A291, P292","M290, A291, P292",kinase-domain alignment


## 7. Replace alignment-derived HRD/DFG/APE with direct motif calls when unambiguous

This gives motif evidence priority over pure residue transfer.

In [ ]:
for motif in ["HRD","DFG","APE"]:
    hits=motif_hits[motif]
    if len(hits)==1:
        st,s=hits[0]
        features[motif]=list(range(st,st+len(s)))

# Catalytic Lys: if an unambiguous VAIK-like hit is present, use its Lys.
vai_hits=motif_hits["VAIK-like"]
if len(vai_hits)==1:
    st,s=vai_hits[0]
    k_offset=s.index("K")
    features["Catalytic Lys"]=[st+k_offset]

summary=[]
for feat,idxs in features.items():
    ul=[]; pl=[]
    for i in idxs:
        ui=domain_index_to_up(i)
        pi=domain_index_to_pdb(i)
        ul.append(up_label(ui))
        pl.append(pdb_label(pi) if pi is not None else "not observed")
    summary.append({
        "Feature":feat,
        "UniProt residue":", ".join(ul),
        "PDB residue":", ".join(pl)
    })

summary_df=pd.DataFrame(summary)
summary_df

,Feature,UniProt residue,PDB residue
0,Catalytic Lys,I144,I144
1,alphaC Glu,N165,N165
2,Gatekeeper,V191,V191
3,Hinge,"M192, E193, L194","M192, E193, L194"
4,HRD,"H236, R237, D238","H236, R237, D238"
5,DFG,"D265, F266, G267","D265, F266, G267"
6,APE,"A291, P292, E293","A291, P292, E293"


## 8. Kinase-topology QC

A conventional kinase should satisfy approximately:

**gly-loop → catalytic Lys → hinge/gatekeeper → HRD → DFG → APE**

This does not prove every assignment, but it is a useful automated sanity check.

In [ ]:
def first(name):
    return features[name][0] if features.get(name) else None

order = ["Catalytic Lys","Gatekeeper","Hinge","HRD","DFG","APE"]
vals = [(x,first(x)) for x in order]
print(vals)

for (a,ai),(b,bi) in zip(vals[:-1], vals[1:]):
    if ai is not None and bi is not None:
        print(f"{a} < {b}: {'PASS' if ai < bi else 'CHECK'}")

[('Catalytic Lys', np.int64(28)), ('Gatekeeper', np.int64(75)), ('Hinge', np.int64(76)), ('HRD', 120), ('DFG', 149), ('APE', 175)]
Catalytic Lys < Gatekeeper: PASS
Gatekeeper < Hinge: PASS
Hinge < HRD: PASS
HRD < DFG: PASS
DFG < APE: PASS


## 9. 3D visualization using only the protein

Bound inhibitor atoms are not used for the assignments below.

In [ ]:
class ProteinOnly(Select):
    def accept_residue(self,residue):
        return residue.id[0]==" "

io=PDBIO()
io.set_structure(structure)
io.save("protein_only.pdb",ProteinOnly())
pdb_text=open("protein_only.pdb").read()

view=py3Dmol.view(width=1000,height=700)
view.addModel(pdb_text,"pdb")
view.setStyle({"chain":CHAIN_ID},{"cartoon":{"color":"lightgray"}})

colors={
    "Catalytic Lys":"red",
    "alphaC Glu":"orange",
    "Gatekeeper":"magenta",
    "Hinge":"cyan",
    "HRD":"yellow",
    "DFG":"green",
    "APE":"blue",
}

for feat,idxs in features.items():
    if feat not in colors:
        continue
    for di in idxs:
        pi=domain_index_to_pdb(di)
        if pi is None:
            continue
        rr=pdb_residues[pi]
        resi=rr["resnum"]
        view.setStyle(
            {"chain":CHAIN_ID,"resi":resi},
            {"stick":{"color":colors[feat]},"cartoon":{"color":colors[feat]}}
        )
        view.addLabel(
            f"{feat}: {pdb_label(pi)}",
            {"position":{"chain":CHAIN_ID,"resi":resi,"atom":"CA"},
             "backgroundColor":"white","fontColor":"black","fontSize":10}
        )

view.zoomTo({"chain":CHAIN_ID})
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# How to use this notebook for another kinase

Usually only change:

```python
PDB_ID = "XXXX"
CHAIN_ID = "A"
```

The notebook then:

- resolves the UniProt accession from RCSB,
- obtains the target kinase-domain annotation from UniProt,
- maps PDB numbering ↔ UniProt numbering,
- detects conserved kinase motifs,
- assigns gatekeeper/hinge from kinase-domain topology,
- shows the residues in 3D.

## If no experimental PDB exists
The same logic can be adapted to an AlphaFold structure by providing:

1. a UniProt accession,
2. the AlphaFold PDB/mmCIF model.

## Why not just search HRD and DFG?
HRD and DFG are excellent anchors, but **hinge and gatekeeper are structural/topological positions**, not unique sequence strings.  
Therefore a robust workflow should combine motif detection with a kinase-specific alignment/profile.

## Stronger research-grade extension
For large-scale kinase analysis, the next step is to replace the single-reference topology transfer with a
**kinase-specific multiple-sequence/profile alignment or KLIFS-style standardized pocket numbering**.
That is preferable when annotating hundreds of kinases, pseudokinases, or highly divergent kinase families.